# 03. Guardrails and Human in the Loop

This notebook shows how to validate outputs and route uncertain cases to a person. That is a core pattern for research workflows where ambiguity matters. The SDK README groups this under guardrails and human in the loop: guardrails check whether the run is safe or valid, and human review steps let a person intervene when the model should not decide alone.

## Why this matters in DH
In digital humanities, uncertainty is not always a failure. It can be part of the evidence. A blurred place name, a conflicting date, or a doubtful transcription often needs scholarly judgment. This notebook shows how to preserve that judgment instead of hiding it behind automation.

The goal is not to eliminate ambiguity. The goal is to make ambiguity visible and actionable.

## Concepts
- Guardrails
- Human review
- Uncertainty thresholds
- Safer automation
- Validation before action
- Evidence-based decision making

Relevant docs: [Guardrails](https://openai.github.io/openai-agents-python/guardrails/) and [Human in the loop](https://openai.github.io/openai-agents-python/human_in_the_loop/).

## Tracing
View traces in the OpenAI Traces dashboard: https://platform.openai.com/traces

## Dataset
The notebook uses a small set of ambiguous archival-style records stored in `data/` and loaded by `notebooks/common.py`.

## Colab data setup
If the notebook is opened in Google Colab without cloning the repo, upload `data.zip` from the repository root and extract it before running the example cells. A setup cell for that is included below.

In [ ]:
from pathlib import Path

candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)

if data_dir is None:
    try:
        from google.colab import files
        import io
        import zipfile

        print('Running in Google Colab and data/ is missing.')
        print('Upload data.zip from the repository root, then this cell will extract it.')
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
        with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as zf:
            zf.extractall('.')
        data_dir = next((path for path in candidate_dirs if path.exists()), None)
        assert data_dir is not None, 'data/ folder was not found after extraction.'
        print('data/ extracted successfully.')
    except ModuleNotFoundError:
        raise FileNotFoundError('data/ folder not found. Clone the repo locally or upload data.zip in Colab.')


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'

from agents import set_tracing_export_api_key
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])

from common import LETTER_TEXTS, ReviewDecision
from agents import Agent, Runner, InputGuardrail, GuardrailFunctionOutput

records = LETTER_TEXTS
records

## Step 1: A tiny review heuristic

We treat low confidence or uncertain OCR as a signal for human review. The heuristic is intentionally simple so the policy is clear before a more complex validation setup.

In [ ]:
def should_review(decision: ReviewDecision) -> bool:
    return decision.uncertain or decision.confidence < 0.8

sample_decision = ReviewDecision(record_id=3, uncertain=True, confidence=0.52, notes='OCR ambiguity around place name')
should_review(sample_decision)

## Step 2: Build a guardrail

A guardrail can reject inputs that clearly need human review before the agent continues. In the SDK, guardrails are a first-class concept rather than an afterthought, which makes them useful for real workflows where you need a policy boundary before the model acts.

You can think of them as automated quality checks that sit in front of or behind the agent.

In [ ]:
def uncertainty_guardrail(ctx, agent, input_text):
    if isinstance(input_text, str) and ('may be' in input_text.lower() or 'unclear' in input_text.lower()):
        return GuardrailFunctionOutput(output_info='uncertain text', tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info='ok', tripwire_triggered=False)

guardrail = InputGuardrail(uncertainty_guardrail, name='uncertainty_guardrail')
guardrail

## Step 3: Agent with guardrail

If the text looks uncertain, the agent should stop and let the person review it. This is the main classroom lesson: a good agentic pipeline knows when not to proceed.

For research settings, that protects against fabricated precision and makes the workflow more trustworthy.

In [ ]:
review_agent = Agent(
    name='Review Agent',
    instructions=(
        'Assess whether the text can be safely processed. '
        'If it is uncertain, return a short note asking for human review.'
    ),
    input_guardrails=[guardrail],
)

review_agent

## Step 4: Run on a clean and an uncertain record

The clean record should pass. The uncertain one should trigger the guardrail. In notebooks, use `await Runner.run(...)` rather than `run_sync(...)` because the kernel already manages an event loop. This is where a normal path can be compared with an interrupted path and show why human review exists.

## Further reading
- Guardrails: https://openai.github.io/openai-agents-python/guardrails/
- Human in the loop: https://openai.github.io/openai-agents-python/human_in_the_loop/
- Tracing: https://openai.github.io/openai-agents-python/tracing/


In [ ]:
from agents import trace

with trace('DH guardrails demo'):
    clean_result = await Runner.run(review_agent, records[0]['text'])
print(clean_result.final_output)

uncertain_text = 'The scan is unclear; the place may be Seville or Sevilla.'
try:
    with trace('DH guardrails uncertain case'):
        uncertain_result = await Runner.run(review_agent, uncertain_text)
    print(uncertain_result.final_output)
except Exception as exc:
    print(type(exc).__name__, exc)

## Human-in-the-loop checkpoint

If the guardrail triggers, the next step is not to force a guess. It is to ask a person to resolve the ambiguity and then resume.

## Exercise

Change the threshold so only records with both OCR ambiguity and low confidence are sent to review.

## Solution

Modify `uncertainty_guardrail` to check for two conditions at once, for example `('unclear' in text and confidence < 0.8)`.